# 01 · Explore G06N (AI / machine learning) in Google Patents

Measures the corpus before we extract it:

1. How many **US** documents carry a G06N code, by year and by grant/application
2. Which G06N **subgroups** are largest, to choose a scope of about 20,000 patent families
3. What the **citation** data looks like, and how many citations point inside the corpus (needed for the examiner-citation test set)
4. How much the **full-text** extraction would cost in BigQuery quota (dry run only)

**Safe by design:** every query is dry-run first, and runs only below a size limit (`max_gb`), so nothing can use more of the free 1 TB monthly quota than allowed. Results are saved to your Google Drive, so later notebooks don't scan the same data again.

Run the cells from top to bottom. When asked, allow access to your Google account (for BigQuery) and to Google Drive.

In [ ]:
PROJECT_ID = "patent-search-515813"
TABLE = "patents-public-data.patents.publications"
DRIVE_DIR = "/content/drive/MyDrive/patent-search"

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

USED_GB = 0.0

def estimate_gb(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=cfg).total_bytes_processed / 1e9

def run(sql, max_gb):
    """Dry-run, then run only if the query scans less than max_gb."""
    global USED_GB
    gb = estimate_gb(sql)
    print(f"Estimated scan: {gb:.1f} GB (limit {max_gb} GB)")
    if gb > max_gb:
        raise RuntimeError("Query skipped: above the size limit. Nothing was charged.")
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=int(max_gb * 1e9))
    df = client.query(sql, job_config=cfg).to_dataframe()
    USED_GB += gb
    print(f"Rows: {len(df):,} | quota used this session: {USED_GB:.1f} GB")
    return df

print("Connected to BigQuery project", PROJECT_ID)

In [ ]:
# Check the table's fields (free: reads metadata only)
schema = {f.name: f for f in client.get_table(TABLE).schema}
for name in ["cpc", "citation"]:
    print(name, "->", [sub.name for sub in schema[name].fields])

## 1 · US G06N documents (metadata only)

Reads only small columns. Saved to Drive as `g06n_us_meta.parquet` for the later extraction notebook.

In [ ]:
SQL_META = f"""
SELECT
  publication_number, family_id, kind_code, application_kind,
  filing_date, priority_date, publication_date, grant_date,
  ARRAY(SELECT c.code FROM UNNEST(cpc) c WHERE c.code LIKE 'G06N%') AS g06n_codes,
  (SELECT c.code FROM UNNEST(cpc) c WHERE c.first LIMIT 1) AS first_cpc
FROM `{TABLE}`
WHERE country_code = 'US'
  AND filing_date >= 20100101
  AND EXISTS (SELECT 1 FROM UNNEST(cpc) c WHERE c.code LIKE 'G06N%')
"""
meta = run(SQL_META, max_gb=150)
meta.to_parquet(f"{DRIVE_DIR}/g06n_us_meta.parquet", index=False)

In [ ]:
import pandas as pd
meta["year"] = meta["filing_date"] // 10000
meta["is_grant"] = meta["kind_code"].str.startswith("B")
meta["g06n_first"] = meta["first_cpc"].fillna("").str.startswith("G06N")

print("US documents with a G06N code (filed 2010+):", f"{len(meta):,}")
print("Distinct patent families:", f"{meta['family_id'].nunique():,}")
print("Families where G06N is the FIRST (main) code:", f"{meta.loc[meta.g06n_first, 'family_id'].nunique():,}")

print("\nDocuments by filing year (grants / applications):")
print(meta.groupby("year")["is_grant"].agg(grants="sum", total="count").assign(
    applications=lambda d: d.total - d.grants)[["grants", "applications", "total"]].to_string())

## 2 · Largest G06N subgroups

Counts patent families per main group (for example G06N3 = neural networks, G06N20 = machine learning) and per subgroup.

In [ ]:
codes = meta[["family_id", "g06n_codes"]].explode("g06n_codes").dropna()
codes["main_group"] = codes["g06n_codes"].str.split("/").str[0]

print("Families per main group:")
print(codes.groupby("main_group")["family_id"].nunique().sort_values(ascending=False).to_string())

print("\nTop 20 subgroups by families:")
print(codes.groupby("g06n_codes")["family_id"].nunique().sort_values(ascending=False).head(20).to_string())

## 3 · Citations of recent G06N documents

Reads the citation list of documents filed 2020 or later, to see which citation categories exist and how many citations point to other documents in the corpus. Saved to Drive as `g06n_us_citations.parquet`.

In [ ]:
SQL_CIT = f"""
SELECT
  p.publication_number, p.filing_date,
  c.publication_number AS cited, c.category, c.type
FROM `{TABLE}` p, UNNEST(p.citation) c
WHERE p.country_code = 'US'
  AND p.filing_date >= 20200101
  AND EXISTS (SELECT 1 FROM UNNEST(p.cpc) x WHERE x.code LIKE 'G06N%')
  AND c.publication_number != ''
"""
cit = run(SQL_CIT, max_gb=250)
cit.to_parquet(f"{DRIVE_DIR}/g06n_us_citations.parquet", index=False)

In [ ]:
corpus_ids = set(meta["publication_number"])
corpus_families = dict(zip(meta["publication_number"], meta["family_id"]))
cit["cited_in_corpus"] = cit["cited"].isin(corpus_ids)

print("Citation category values (top 15):")
print(cit["category"].fillna("(empty)").value_counts().head(15).to_string())
print("\nCitation type values:")
print(cit["type"].fillna("(empty)").value_counts().head(10).to_string())

print(f"\nCitations from 2020+ documents: {len(cit):,}")
print(f"... pointing to US G06N documents in the corpus: {cit['cited_in_corpus'].sum():,} ({cit['cited_in_corpus'].mean():.0%})")

per_doc = cit[cit.cited_in_corpus].groupby("publication_number")["cited"].nunique()
print(f"Documents (filed 2020+) with at least 1 in-corpus citation: {len(per_doc):,}")
print(f"... with at least 3: {(per_doc >= 3).sum():,}")

## 4 · Cost of the full-text extraction (dry run only, nothing is run)

Estimates how much quota reading the title, abstract, claims and description would use. BigQuery bills by whole columns read, so the size is roughly the same whatever the filter.

In [ ]:
def text_sql(columns):
    return f"""
SELECT publication_number, {columns}
FROM `{TABLE}`
WHERE country_code = 'US' AND filing_date >= 20100101
  AND EXISTS (SELECT 1 FROM UNNEST(cpc) c WHERE c.code LIKE 'G06N%')
"""

for label, cols in [
    ("title + abstract", "title_localized, abstract_localized"),
    ("+ claims", "title_localized, abstract_localized, claims_localized"),
    ("+ claims + description", "title_localized, abstract_localized, claims_localized, description_localized"),
]:
    print(f"{label:28} {estimate_gb(text_sql(cols)):8.1f} GB")

In [ ]:
print("Quota used in this session:", f"{USED_GB:.1f} GB of the free 1,000 GB per month")
print("Saved to Drive:", os.listdir(DRIVE_DIR))